In [270]:
from taxipred.utils.constants import TAXI_CSV_PATH
import pandas as pd

df = pd.read_csv(TAXI_CSV_PATH)

df

,Trip_Distance_km,Time_of_Day,Day_of_Week,Passenger_Count,Traffic_Conditions,Weather,Base_Fare,Per_Km_Rate,Per_Minute_Rate,Trip_Duration_Minutes,Trip_Price
0,19.35,Morning,Weekday,3.0,Low,Clear,3.56,0.80,0.32,53.82,36.2624
1,47.59,Afternoon,Weekday,1.0,High,Clear,NaN,0.62,0.43,40.57,NaN
2,36.87,Evening,Weekend,1.0,High,Clear,2.70,1.21,0.15,37.27,52.9032
3,30.33,Evening,Weekday,4.0,Low,NaN,3.48,0.51,0.15,116.81,36.4698
4,NaN,Evening,Weekday,3.0,High,Clear,2.93,0.63,0.32,22.64,15.6180
...,...,...,...,...,...,...,...,...,...,...,...
995,5.49,Afternoon,Weekend,4.0,Medium,Clear,2.39,0.62,0.49,58.39,34.4049
996,45.95,Night,Weekday,4.0,Medium,Clear,3.12,0.61,NaN,61.96,62.1295
997,7.70,Morning,Weekday,3.0,Low,Rain,2.08,1.78,NaN,54.18,33.1236
998,47.56,Morning,Weekday,1.0,Low,Clear,2.67,0.82,0.17,114.94,61.2090


In [271]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       950 non-null    float64
 1   Time_of_Day            950 non-null    object 
 2   Day_of_Week            950 non-null    object 
 3   Passenger_Count        950 non-null    float64
 4   Traffic_Conditions     950 non-null    object 
 5   Weather                950 non-null    object 
 6   Base_Fare              950 non-null    float64
 7   Per_Km_Rate            950 non-null    float64
 8   Per_Minute_Rate        950 non-null    float64
 9   Trip_Duration_Minutes  950 non-null    float64
 10  Trip_Price             951 non-null    float64
dtypes: float64(7), object(4)
memory usage: 86.1+ KB


In [272]:
df_clean = df.dropna().copy()

In [273]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Time_of_Day            562 non-null    object 
 2   Day_of_Week            562 non-null    object 
 3   Passenger_Count        562 non-null    float64
 4   Traffic_Conditions     562 non-null    object 
 5   Weather                562 non-null    object 
 6   Base_Fare              562 non-null    float64
 7   Per_Km_Rate            562 non-null    float64
 8   Per_Minute_Rate        562 non-null    float64
 9   Trip_Duration_Minutes  562 non-null    float64
 10  Trip_Price             562 non-null    float64
dtypes: float64(7), object(4)
memory usage: 52.7+ KB


In [274]:
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
df_clean_num = df_clean[numeric_cols].copy()
df_clean_num.info()

<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Passenger_Count        562 non-null    float64
 2   Base_Fare              562 non-null    float64
 3   Per_Km_Rate            562 non-null    float64
 4   Per_Minute_Rate        562 non-null    float64
 5   Trip_Duration_Minutes  562 non-null    float64
 6   Trip_Price             562 non-null    float64
dtypes: float64(7)
memory usage: 35.1 KB


In [275]:
import numpy as np

mask_matrix = pd.DataFrame(
    np.random.rand(*df_clean_num.shape) < 0.05,
    index=df_clean_num.index,
    columns=df_clean_num.columns
)

df_masked_num = df_clean_num.mask(mask_matrix)

print("Expected missing per column ~", int(round(0.05 * len(df_clean_num))))
print("\nActual missing per column:")
display(df_masked_num.isna().sum())


Expected missing per column ~ 28

Actual missing per column:


Trip_Distance_km         30
Passenger_Count          31
Base_Fare                32
Per_Km_Rate              25
Per_Minute_Rate          33
Trip_Duration_Minutes    28
Trip_Price               28
dtype: int64

In [276]:
from sklearn.impute import SimpleImputer

mean_imputer = SimpleImputer(strategy="mean")
df_mean = pd.DataFrame(
    mean_imputer.fit_transform(df_masked_num),
    columns=df_masked_num.columns,
    index=df_masked_num.index
)

df_mean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Passenger_Count        562 non-null    float64
 2   Base_Fare              562 non-null    float64
 3   Per_Km_Rate            562 non-null    float64
 4   Per_Minute_Rate        562 non-null    float64
 5   Trip_Duration_Minutes  562 non-null    float64
 6   Trip_Price             562 non-null    float64
dtypes: float64(7)
memory usage: 35.1 KB


In [277]:
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

linear_imputer = IterativeImputer(
    estimator=LinearRegression(),
    max_iter=1,               # key: one pass (linear regression baseline)
    random_state=42,
    sample_posterior=False
)

df_linear = pd.DataFrame(
    linear_imputer.fit_transform(df_masked_num),
    columns=df_masked_num.columns,
    index=df_masked_num.index
)

df_linear.info()


<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Passenger_Count        562 non-null    float64
 2   Base_Fare              562 non-null    float64
 3   Per_Km_Rate            562 non-null    float64
 4   Per_Minute_Rate        562 non-null    float64
 5   Trip_Duration_Minutes  562 non-null    float64
 6   Trip_Price             562 non-null    float64
dtypes: float64(7)
memory usage: 35.1 KB


/home/apollonspc/taxi-prediction-fullstack-omer-aytug/.venv/lib/python3.14/site-packages/sklearn/impute/_iterative.py:867: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [278]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

mice_imputer = IterativeImputer(
    estimator=LinearRegression(),
    random_state=42,
    max_iter=20,
    sample_posterior=False
)

df_mice = pd.DataFrame(
    mice_imputer.fit_transform(df_masked_num),
    columns=df_masked_num.columns,
    index=df_masked_num.index
)

df_mice.info()

<class 'pandas.core.frame.DataFrame'>
Index: 562 entries, 0 to 998
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Trip_Distance_km       562 non-null    float64
 1   Passenger_Count        562 non-null    float64
 2   Base_Fare              562 non-null    float64
 3   Per_Km_Rate            562 non-null    float64
 4   Per_Minute_Rate        562 non-null    float64
 5   Trip_Duration_Minutes  562 non-null    float64
 6   Trip_Price             562 non-null    float64
dtypes: float64(7)
memory usage: 35.1 KB


In [279]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Only evaluate at positions we intentionally masked
y_true   = df_clean_num.values[mask_matrix.values]
y_mean   = df_mean.values[mask_matrix.values]
y_linear = df_linear.values[mask_matrix.values]
y_mice   = df_mice.values[mask_matrix.values]

def rmse(y_t, y_p):
    return np.sqrt(mean_squared_error(y_t, y_p))

def mae(y_t, y_p):
    return mean_absolute_error(y_t, y_p)

results = pd.DataFrame({
    "RMSE": [rmse(y_true, y_mean), rmse(y_true, y_linear), rmse(y_true, y_mice)],
    "MAE":  [mae(y_true, y_mean),  mae(y_true, y_linear),  mae(y_true, y_mice)]
}, index=["Mean", "LinearRegression", "MICE"])

display(results.sort_values("RMSE"))



,RMSE,MAE
MICE,13.852487,6.536096
LinearRegression,13.876751,6.637809
Mean,18.521962,9.562076


In [280]:
var_compare = pd.DataFrame({
    "True (df_clean_num)": df_clean_num.var(),
    "Mean Imputed": df_mean.var(),
    "Linear Imputed": df_linear.var(),
    "MICE": df_mice.var()
})

display(var_compare)


,True (df_clean_num),Mean Imputed,Linear Imputed,MICE
Trip_Distance_km,447.456825,421.090858,437.961967,438.126022
Passenger_Count,1.229693,1.166407,1.169096,1.169252
Base_Fare,0.758784,0.717873,0.718346,0.718350
Per_Km_Rate,0.185202,0.177149,0.178174,0.179206
Per_Minute_Rate,0.013187,0.012492,0.012624,0.012629
Trip_Duration_Minutes,1032.236406,967.540481,974.739660,982.768368
Trip_Price,1932.370937,1896.995160,1955.334411,1957.332821


In [281]:
best_method = results["RMSE"].idxmin()
print("Best method by RMSE:", best_method)


Best method by RMSE: MICE
